<a href="https://colab.research.google.com/github/Munjiwon/SpecialTopics-in-TextMining/blob/master/ch03/word2vec_gensim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3주차 실습 1 — gensim 으로 CBOW 와 Skip-gram 학습

**이 노트북의 새 개념**: 같은 말뭉치로 두 구조(sg=0 CBOW, sg=1 Skip-gram)를 학습해 유사어를 비교한다.
Negative Sampling(negative=5)과 서브샘플링(sample=1e-3)은 gensim 기본 설정을 쓴다.

In [1]:
%pip install -q gensim kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 10.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 95.4 MB/s eta 0:00:00


In [ ]:
import sys, torch
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

In [3]:
# 말뭉치 준비와 형태소 토큰화 — 1주차와 같은 Steam 리뷰 감성 말뭉치("레이블<TAB>텍스트")를 내려받고,
# 단어 임베딩 학습에는 문장만 쓰므로(레이블은 따로 보관만 한다) 한국어는 형태소 단위로 나눈다(2주차에서 본 것처럼 형태소 단위가 유리하다).
STEAM_URL = "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/steam.txt"
CORPUS_PATH = "data/steam.txt"  # code03/data/ 에 저장 (노트북 위치 기준 상대경로)
USE_OWN_CORPUS = False
if USE_OWN_CORPUS:
    from google.colab import files
    uploaded = files.upload()
    CORPUS_PATH = "/content/" + next(iter(uploaded))
else:
    import os, urllib.request
    os.makedirs(os.path.dirname(CORPUS_PATH), exist_ok=True)
    if not os.path.exists(CORPUS_PATH):  # 이미 내려받았으면 건너뜀
        urllib.request.urlretrieve(STEAM_URL, CORPUS_PATH)

# 파일 형식: 한 줄에 리뷰 하나, "레이블(0=부정, 1=긍정)<TAB>리뷰 문장"
lines, labels = [], []
with open(CORPUS_PATH, encoding="utf-8") as f:
    for line in f:
        label, text = line.rstrip("\n").split("\t", 1)   # 첫 탭을 기준으로 두 칸으로 나눔
        if text.strip():
            labels.append(int(label))
            lines.append(text.strip())
print(f"리뷰 {len(lines):,}개 (부정 {labels.count(0):,} / 긍정 {labels.count(1):,})")

from kiwipiepy import Kiwi
kiwi = Kiwi()
# 문장(줄) 하나를 토큰 목록 하나로 — gensim 과 직접 구현 모두 이 형태를 쓴다
sentences = [[t.form for t in kiwi.tokenize(l)] for l in lines]
print(f"문장 {len(sentences):,}개, 토큰 {sum(map(len, sentences)):,}개")

리뷰 100,000개 (부정 50,004 / 긍정 49,996)
문장 100,000개, 토큰 2,548,479개


In [4]:
import gensim
from gensim.models import Word2Vec
print("gensim", gensim.__version__)

common = dict(vector_size=100, window=5, min_count=3, negative=5, sample=1e-3, epochs=10, seed=42, workers=1)
cbow = Word2Vec(sentences, sg=0, **common)      # 주변 단어 평균으로 중심 단어 예측
sg   = Word2Vec(sentences, sg=1, **common)      # 중심 단어로 주변 단어 예측
print("어휘 크기:", len(sg.wv))

gensim 4.4.0
어휘 크기: 15492


학습 중 `Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'` 이 몇 줄 찍힐 수 있다.
gensim 4.4.0 이 BLAS 내적 함수를 `except -1` 로 선언해, 내적이 정확히 −1.0 이면 오류로 오인하고 그 내적을 0 으로 바꾸는 문제다.
수백만 번의 갱신 중 몇 번뿐이라 결과에 주는 영향은 무시할 만하다.

## 유사어 비교
질의어는 말뭉치에 자주 나오는 일반명사(NNG) 3개로 고른다. 조사·어미는 어디에나 나오므로 이웃이 의미를 갖지 않는다.

In [5]:
from collections import Counter
nouns = Counter(t.form for l in lines for t in kiwi.tokenize(l) if t.tag == "NNG" and len(t.form) > 1)
queries = [w for w, _ in nouns.most_common(40) if w in sg.wv][:3]   # 자주 나오는 일반명사 3개
for q in queries:
    print(f"[{q}]")
    print("  CBOW     :", [w for w, _ in cbow.wv.most_similar(q, topn=5)])
    print("  Skip-gram:", [w for w, _ in sg.wv.most_similar(q, topn=5)])

[게임]
  CBOW     : ['작품', '겜', '것', '상황', '프로그램']
  Skip-gram: ['케주얼', '겜', '머시나리움', '리듬겜', '떡신']
[플레이]
  CBOW     : ['어택', '현자', '클리어', '쿨', '즐기']
  Skip-gram: ['플래이', '앤딩', '시큼', '플레잉', '즐기']
[재미]
  CBOW     : ['깊이', '제미', '의미', '맛', '가치']
  Skip-gram: ['제미', '메리트', '볼륨감', '감흥', '깊이']


## 모델 저장
다음 노트북(시각화)이나 4주차에서 다시 쓰려면 Google Drive 에 저장해 둔다.

In [6]:
sg.save("w2v_sg.model")
print("저장: w2v_sg.model (Colab 에서는 /content 아래 — 세션이 끝나면 지워지므로 Drive 로 옮긴다)")

저장: w2v_sg.model (Colab 에서는 /content 아래 — 세션이 끝나면 지워지므로 Drive 로 옮긴다)


## 직접 해 보기
1. `window` 를 2 와 10 으로 바꿔 유사어가 "대체 가능한 단어"에서 "같은 주제 단어"로 바뀌는지 보자.
2. 저빈도 단어를 질의로 넣으면 CBOW 와 Skip-gram 중 어느 쪽 결과가 그럴듯한가?
3. `negative=0, hs=1` (계층적 소프트맥스)로 바꾸면 학습 시간과 결과가 어떻게 달라지는가?